# Run Pipeline

This notebook follows the current local workflow:

1. Extract BTC archives into `data/keyframes/` and `data/map-keyframes/`
2. Import metadata into `data/index/metadata.jsonl`
3. Train LoRA CLIP by session, resuming between sessions
4. Extract CLIP features for the keyframes
5. Build the local two-level FAISS index
6. Run a local search test and display the results

In [1]:
from pathlib import Path
import os

def _discover_project_root() -> Path:
    env_root = os.getenv('AIC_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser()
        if candidate.exists():
            return candidate.resolve()

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'backend').is_dir():
            return candidate.resolve()

    raise RuntimeError('Set AIC_PROJECT_ROOT or open the notebook inside the project folder.')

PROJECT_ROOT = _discover_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: D:\AI\AIC 2026\video-search-agent


In [2]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', '-r', 'D:\\AI\\AIC 2026\\video-search-agent\\requirements.txt'], returncode=0)

## Step 0 - Check the runtime

Use this cell to confirm whether the notebook is attached to an NVIDIA GPU or running on CPU.

In [3]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

PyTorch: 2.4.1+cpu
CUDA available: False
Device: cpu


## Step 1 - Extract BTC archives

Run this only once per dataset update. It creates the raw `keyframes/` and `map-keyframes/` folders that the rest of the pipeline reads.

In [4]:
# Preview what will be extracted
subprocess.run([sys.executable, 'scripts/extract_btc_data.py', '--dry-run'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/extract_btc_data.py', '--dry-run'], returncode=0)

In [5]:
# Extract for real
subprocess.run([sys.executable, 'scripts/extract_btc_data.py'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/extract_btc_data.py'], returncode=0)

## Step 2 - Import metadata

This writes the canonical `data/index/metadata.jsonl` used by training, feature extraction, and indexing.

In [6]:
# Add --with-transcript if you want Whisper transcript assignment
subprocess.run([sys.executable, 'scripts/import_btc_data.py', '--with-transcript'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/import_btc_data.py', '--with-transcript'], returncode=0)

## Step 3 - Train LoRA by session

Train in short sessions, then resume from the saved checkpoint. That keeps each run manageable and makes it easy to continue later.

Suggested pattern:
- Session 1: train on a smaller subset or fewer epochs
- Session 2+: resume from `data/index/lora_weights.pt`
- Increase `--limit` only when the previous session is stable

In [8]:
import torch

TRAIN_LIMIT = 0
TEST_LIMIT = 100
TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
TEST_NUM_WORKERS = 0
FEATURE_BATCH_SIZE = 16
FEATURE_TEST_LIMIT = 100
TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Training settings:')
print(f'  limit={TRAIN_LIMIT} (full train)')
print(f'  test_limit={TEST_LIMIT} (smoke test)')
print(f'  epochs={TRAIN_EPOCHS}')
print(f'  batch_size={TRAIN_BATCH_SIZE}')
print(f'  num_workers={TRAIN_NUM_WORKERS}')
print(f'  device={TRAIN_DEVICE}')
print(f'  feature_batch_size={FEATURE_BATCH_SIZE}')

Training settings:
  limit=0 (full train)
  test_limit=100 (smoke test)
  epochs=1
  batch_size=32
  num_workers=0
  device=cpu
  feature_batch_size=16


In [9]:
# Session 0: TEST
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TEST_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TEST_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


Running: c:\Users\Administrator\AppData\Local\Programs\Python\Python312\python.exe scripts/train_lora_clip.py --limit 100 --epochs 1 --batch-size 32 --num-workers 0 --device cpu


CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', 'scripts/train_lora_clip.py', '--limit', '100', '--epochs', '1', '--batch-size', '32', '--num-workers', '0', '--device', 'cpu'], returncode=0)

In [ ]:
# Session 1: fresh training run
# Increase --epochs if you want a longer first session.
import subprocess, sys

train_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Session 2: resume from the saved checkpoint
# Run this after Session 1 if you want to continue training in another pass.
import subprocess, sys

resume_cmd = [
    sys.executable,
    'scripts/train_lora_clip.py',
    '--limit', str(TRAIN_LIMIT),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--device', TRAIN_DEVICE,
    '--resume',
]
print('Running:', ' '.join(resume_cmd))
subprocess.run(resume_cmd, cwd=PROJECT_ROOT, check=True)


## Step 4 - Extract CLIP features

This reads your current keyframes and writes `.npy` vectors to `data/clip-features/`.

In [ ]:
!python scripts/extract_clip_features.py --num-workers 4 --batch-size 128

### Bước 4.1: Chạy Thử Nghiệm (Test 100 Keyframes)
Dùng ô này nếu bạn chỉ muốn trích xuất thử 100 ảnh để kiểm tra tốc độ hoặc debug.

In [10]:
!python scripts/extract_clip_features.py --num-workers 4 --batch-size 128 --limit 100

2026-08-17 11:47:23,170 - INFO - Found 100 pending keyframes. Loading CLIP model on privateuseone:0...
2026-08-17 11:47:26,588 - INFO - Injected LoRA (rank=4, alpha=1.0) into 24 layers. Target: ['out_proj', 'c_proj']
2026-08-17 11:47:26,596 - INFO - Loaded LoRA weights (rank=4, alpha=1.0) from D:\AI\AIC 2026\video-search-agent\data\index\lora_weights.pt
2026-08-17 11:47:26,598 - INFO - Loaded LoRA-CLIP from D:\AI\AIC 2026\video-search-agent\data\index\lora_weights.pt (rank=4, alpha=1.0)

Extracting Features (Batched): 100%|██████████| 100/100 [00:07<00:00, 14.20it/s]
2026-08-17 11:47:33,644 - INFO - Successfully extracted features for 100/100 keyframes.
2026-08-17 11:47:33,644 - INFO - Features saved to: D:\AI\AIC 2026\video-search-agent\data\clip-features


## Step 5 - Build the local FAISS index

This creates `data/index/video.index` for keyframe lookup and `data/index/scene.index` for coarse filtering.

In [11]:
subprocess.run([sys.executable, '-m', 'backend.embedding.build_index'], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['c:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'backend.embedding.build_index'], returncode=0)

## Step 6 - Local search test

This is the notebook-style test flow similar to the old version: encode a query, search the local FAISS index, and print the top hits.

In [12]:
import json
from pathlib import Path

import faiss
import numpy as np

from backend.config import FAISS_INDEX_PATH, FAISS_METADATA_PATH
from backend.embedding.clip_encoder import encode_text

In [14]:
index = faiss.read_index(str(FAISS_INDEX_PATH))
with open(FAISS_METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Loaded {index.ntotal} vectors and {len(metadata)} metadata rows.')

Loaded 100 vectors and 100 metadata rows.


In [15]:
query = 'a photo of a tree'
vec = encode_text(query).reshape(1, -1).astype(np.float32)
faiss.normalize_L2(vec)

top_k = 10
scores, indices = index.search(vec, top_k)

print(f"Query: {query}")
for i, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    if idx < 0 or idx >= len(metadata):
        continue
    item = metadata[idx]
    print(f"#{i:02d} | score={score:.4f} | {item.get('video_id', '')} | frame={item.get('frame_id', '')} | pts={item.get('pts_time', 0.0):.2f}s")

Query: a photo of a tree
#01 | score=0.2413 | L21_V001 | frame=2724 | pts=90.80s
#02 | score=0.2261 | L21_V001 | frame=4836 | pts=161.20s
#03 | score=0.2137 | L21_V001 | frame=2820 | pts=94.00s
#04 | score=0.2134 | L21_V001 | frame=4998 | pts=166.60s
#05 | score=0.2097 | L21_V001 | frame=7134 | pts=237.80s
#06 | score=0.2090 | L21_V001 | frame=2376 | pts=79.20s
#07 | score=0.2059 | L21_V001 | frame=3408 | pts=113.60s
#08 | score=0.2016 | L21_V001 | frame=366 | pts=12.20s
#09 | score=0.2014 | L21_V001 | frame=4188 | pts=139.60s
#10 | score=0.2008 | L21_V001 | frame=3348 | pts=111.60s


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

n = min(top_k, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f"Search results: {query}")

for i, ax in enumerate(axes.flat):
    if i >= n:
        ax.axis('off')
        continue
    idx = indices[0][i]
    if idx < 0 or idx >= len(metadata):
        ax.axis('off')
        continue

    item = metadata[idx]
    img_path = Path(item.get('path', ''))
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert('RGB'))
    else:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)

    ax.set_title(f"#{i+1} | {item.get('video_id', '')} | {item.get('frame_id', '')}")
    ax.axis('off')

plt.tight_layout()
plt.show()